# K-Modes Clustering on the Mushroom Dataset

This notebook recreates the local K-Modes demo in a Google Colab-friendly format.

It uses the UCI Mushroom dataset, which is composed of categorical attributes. Because the data are not numeric, the standard K-Means algorithm is not appropriate here; instead we use `KModes`, which clusters by matching categorical values and measuring dissimilarity with Hamming distance.

> The elbow method is a heuristic, not a definitive answer. Use your domain knowledge and critical thinking to choose the final number of clusters.

In [ ]:
!pip install -q pandas matplotlib kmodes

print('Packages installed.')

## Imports and setup

We import the clustering model and plotting libraries, then suppress a known Matplotlib warning so the notebook stays clean during classroom use.

In [ ]:
import warnings

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from kmodes.kmodes import KModes

warnings.filterwarnings(
    'ignore',
    message=r'The resize_event function was deprecated.*',
    category=matplotlib.MatplotlibDeprecationWarning,
)

print('Libraries imported.')

## Load the mushroom dataset

The original UCI dataset is downloaded from the public archive. The first column is the target label (`edible` vs `poisonous`), which we remove for unsupervised clustering.

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data'
columns = [
    'target',
    'cap-shape',
    'cap-surface',
    'cap-color',
    'bruises',
    'odor',
    'gill-attachment',
    'gill-spacing',
    'gill-size',
    'gill-color',
    'stalk-shape',
    'stalk-root',
    'stalk-surface-above-ring',
    'stalk-surface-below-ring',
    'stalk-color-above-ring',
    'stalk-color-below-ring',
    'veil-type',
    'veil-color',
    'ring-number',
    'ring-type',
    'spore-print-color',
    'population',
    'habitat',
]

df = pd.read_csv(url, header=None, names=columns)

# Drop target label for unsupervised learning, and compress row count for fast classroom execution
X = df.drop(columns=['target']).head(1000)

print('Dataset shape:', df.shape)
print('Working data shape:', X.shape)
print(X.head())

## Evaluate the elbow

We test a range of cluster counts from 1 to 5 and compare the total clustering cost. The elbow point gives a heuristic signal for choosing a plausible value of `k`.

In [ ]:
cost = []
k_range = range(1, 6)

for k in k_range:
    # 'Cao' initialization is highly recommended over random for K-Modes
    km = KModes(n_clusters=k, init='Cao', n_init=1, verbose=0)
    km.fit(X)
    cost.append(km.cost_)  # Cost = total number of feature mismatches (Hamming distance)

print('Clustering cost for k = 1 to 5:')
for k_value, score in zip(k_range, cost):
    print(f'k={k_value}: cost={score}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_range, cost, marker='o', linestyle='--', color='purple')
plt.title('K-Modes Elbow Method (Mushroom Data)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Clustering Cost (Total Mismatches)')
plt.xticks(k_range)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.show()

## Fit the final model

The script chooses `k=2` because this dataset is naturally binary in many cases: edible vs poisonous. The final model identifies the most common category for each feature in each cluster.

In [ ]:
final_km = KModes(n_clusters=2, init='Cao', n_init=1, verbose=0)
clusters = final_km.fit_predict(X)

print('\n--- Discovered Cluster Modes (Most Frequent Categories) ---')
centroids_df = pd.DataFrame(final_km.cluster_centroids_, columns=X.columns)
print(centroids_df[['cap-shape', 'cap-color', 'odor', 'habitat']])